In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
project_root = Path(__file__).resolve().parents[0] if '__file__' in globals() else Path().resolve().parents[0]
sys.path.insert(0, str(project_root))
pd.set_option('display.max_columns', None)

# Stratified Downsampling (800 samples) 

In [2]:
from tomato.utils import load_critic_review_df, load_movie_df, tomato_data_path

df_critic = load_critic_review_df()
df_movie = load_movie_df()

# merge critic-level data and movie-level data 
df_full = df_critic.merge(
    df_movie,
    on='rotten_tomatoes_link',
    how='inner'
).dropna(
    subset=['review_content']
)

In [3]:
# identify "pure" drama vs "pure" comedy 
df_full['drama_or_comedy'] = np.where(
    (
        df_full['genres'].str.contains('Drama', na=False) & 
        ~df_full['genres'].str.contains('Comedy', na=False)
    ),
    'Drama',
    np.where(
        (
            df_full['genres'].str.contains('Comedy', na=False) & 
            ~df_full['genres'].str.contains('Drama', na=False)
        ),
        'Comedy',
        None
    )
)

# get two most prolific critics 
top_two_critics = df_full.critic_name.value_counts()[:2].index.tolist()

In [4]:
# stratified random sample, 800 samples spready across 16 buckets 9
df_strat = df_full[
    df_full.critic_name.isin(top_two_critics) &
    df_full.content_rating.isin(['R', 'PG']) &
    df_full.drama_or_comedy.notna()
].groupby(
    ['critic_name', 'content_rating', 'drama_or_comedy', 'review_type']
).sample(
    50, random_state=12
).reset_index(
    drop=True
)
# persist the results
df_strat.to_csv(tomato_data_path() / 'df_strat800.csv', index=False)
# view results
df_strat.groupby(
    ['critic_name', 'content_rating', 'drama_or_comedy']
).review_type.value_counts()

critic_name      content_rating  drama_or_comedy  review_type
Dennis Schwartz  PG              Comedy           Fresh          50
                                                  Rotten         50
                                 Drama            Fresh          50
                                                  Rotten         50
                 R               Comedy           Fresh          50
                                                  Rotten         50
                                 Drama            Fresh          50
                                                  Rotten         50
Roger Ebert      PG              Comedy           Fresh          50
                                                  Rotten         50
                                 Drama            Fresh          50
                                                  Rotten         50
                 R               Comedy           Fresh          50
                                                  Rott

In [ ]:
from tomato.encoding import bert_encode_reviews
# encode using bert 
texts = df_strat.review_content.astype(str).tolist()
ids = df_strat.index.tolist()
df_enc = bert_encode_reviews(texts, ids, "bert-base-uncased")
# persist the encoding 
df_enc.to_parquet(tomato_data_path() / 'encoding_strat800.parquet', compression='snappy')